# **VLMs evaluation**

Here, we're going to evaluate the end-to-end performance on the resume parsing task of:
* `GPT-4.1-mini`
* `Gemini-2.5-pro`
* `Gemini-2.5-flash`
* `LiquidAI`
* `Ministral-3:8b`
* RChilli service

**Two evaluation stages:**

**Stage 1: Single-Image Analysis:**
* Independent page processing without cross-page context (218 images = 218 resumes)
* Baseline accuracy metrics per image

**Stage 2: Multi-Image Contextual Evaluation:**
* 112 complete resumes (218 images total)
* Models process entire documents with page-to-page context
* Results reported per resume (not per image)


In [ ]:
from collections import defaultdict
from PIL import Image
import re
import os
from pathlib import Path
from PIL import Image
import json
import torch
from utils import pydantic_schema
import pandas as pd
import base64
from openai import OpenAI
from google import genai
from dotenv import load_dotenv
from utils.ocr_output_norma import JsonDotsOCRNormalization

In [ ]:
resume_images_directory = "data/resumes_images"

In [ ]:
load_dotenv()
API_KEY_OPENAI = os.getenv("OPENAI_API_KEY")
API_KEY_GEMINI = os.getenv("GEMINI_API_KEY")
client_openai = OpenAI(api_key=API_KEY_OPENAI)
client_gemini = genai.Client(api_key=API_KEY_GEMINI)

INPUT_DIR_CONTEXT = Path(resume_images_directory)

OUTPUT_DIR_OPENAI= Path(output_dir_openai)
OUTPUT_DIR_GEMINI=Path(output_dir_gemini)

In [ ]:
def image_to_data_url(path: Path) -> str:
    mime = "image/png" if path.suffix.lower() == ".png" else "image/jpeg"
    data = path.read_bytes()
    b64 = base64.b64encode(data).decode("utf-8")
    return f"data:{mime};base64,{b64}"

def encode_image(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode('utf-8')

## **First evaluation stage**

In [ ]:
PROMPT = """
Extract structured information from this resume image.
Return ONLY the extracted information in JSON format following the provided schema.
Do NOT translate, rewrite, or summarize any text: extract exactly as shown.
Maintain the original language for all extracted text.
Do not hallucinate any information not visible in the image.
If a field is not present, return an empty string or empty list as appropriate.
"""

In [ ]:
from utils.pydantic_schema import ResumeData # pydantic output schema

### **Gemini**

Same code works for `gemini-2.5-pro` and `gemini-2.5-flash`. You can change the model in the `model="model_name"` parameter

In [ ]:
all_images = sorted(
    p for p in INPUT_DIR_CONTEXT.iterdir()
    if p.suffix.lower() in [".png", ".jpg", ".jpeg"]
)

print(f"Processing {len(all_images)} images...")

for img_path in all_images:
    try:
        print(f"→ Processing {img_path.name} ...")

        img = Image.open(img_path).convert("RGB")

        response = client_gemini.models.generate_content(
            model="gemini-2.5-pro",
            contents=[PROMPT, img],
            config={
                "temperature": 0,
                "max_output_tokens": 5000,
                "response_mime_type": "application/json",
                "response_json_schema": ResumeData.model_json_schema()
            },
        )
        resume_data = ResumeData.model_validate_json(response.text)
        output_path = OUTPUT_DIR_GEMINI / f"{img_path.stem}.json"
        json_data = resume_data.model_dump()
        with open(output_path, "w", encoding="utf-8") as f:
            json.dump(json_data, f, indent=2, ensure_ascii=False)

    except Exception as e:
        print(f"Error processing {img_path.name}: {e}")

print("\nDONE!")

### **OpenAI**

`gpt-4.1-mini`

In [ ]:
all_images = sorted(
    p for p in INPUT_DIR_CONTEXT.iterdir()
    if p.suffix.lower() in [".png", ".jpg", ".jpeg"]
)

print(f"Processing {len(all_images)} images...")

for img_path in all_images:
    try:
        print(f"→ Processing {img_path.name} ...")
        base64_data_url = image_to_data_url(img_path)

        response_openai = client_openai.responses.parse(
        model="gpt-4.1-mini",
        input=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "input_text",
                        "text": PROMPT
                    },
                    {
                        "type": "input_image",
                        "image_url": base64_data_url
                    }
                ]
            }
        ], 
        text_format=ResumeData)

        resume_data_openai = response_openai.output_parsed
        output_path_openai = OUTPUT_DIR_OPENAI / f"{img_path.stem}.json"
        json_data_openai = resume_data_openai.model_dump()
        with open(output_path_openai, "w", encoding="utf-8") as f:
            json.dump(json_data_openai, f, indent=2, ensure_ascii=False)

    except Exception as e:
        print(f"Error processing {img_path.name}: {e}")

print("\nDONE!")

## **Second Evaluation Stage**


In [ ]:
def group_cv_pages(image_paths):
    groups = defaultdict(list)
    pattern = re.compile(r"(.*)_page_\d+", re.IGNORECASE)

    for p in image_paths:
        match = pattern.match(p.stem)
        if not match:
            continue
        cv_id = match.group(1)
        groups[cv_id].append(p)

    for cv_id in groups:
        groups[cv_id] = sorted(groups[cv_id])

    return groups

In [ ]:
prompt_context_between_pages = """
You are given multiple images that together form a single resume (CV).
The pages are provided in order and must be interpreted as ONE document.

Extract structured information from the COMPLETE resume using ALL pages.
Information may appear on any page.

Rules:
- Return ONLY valid JSON following the provided schema.
- Do NOT translate, rewrite, or summarize text.
- Preserve the original language exactly as shown.
- Do NOT hallucinate or infer missing information.
- Do NOT duplicate entries across pages.
- If a section continues across pages, merge it correctly.

The output must represent the full resume as a single entity.
"""


In [ ]:
all_images = sorted(
    p for p in INPUT_DIR_CONTEXT.iterdir()
    if p.suffix.lower() in [".png", ".json", ".jpeg"]
)

cv_groups = group_cv_pages(all_images)

print(f"Found {len(cv_groups)} CVs")


### **Gemini**

In [ ]:
for cv_name, pages in cv_groups.items():
    try:
        print(f"\n→ Processing CV: {cv_name} ({len(pages)} pages)")
        images = [Image.open(p).convert("RGB") for p in pages]
        contents = [prompt_context_between_pages] + images
        max_tokens = min(12000, 4000 + 2000 * len(pages))
        response = client_gemini.models.generate_content(
            model="gemini-2.5-pro",
            contents=contents,
            config={
                "temperature": 0,
                "max_output_tokens": max_tokens,
                "response_mime_type": "application/json",
                "response_json_schema": ResumeData.model_json_schema(),
            },
        )
        resume_data = ResumeData.model_validate_json(response.text)
        output_path = OUTPUT_DIR_GEMINI / f"{cv_name}.json"
        with open(output_path, "w", encoding="utf-8") as f:
            json.dump(resume_data.model_dump(), f, indent=2, ensure_ascii=False)
        print(f"Saved {output_path.name}")

        # Token usage
        usage = response.usage_metadata
        print(f"--- TOKEN USAGE ---")
        print(f"  Input tokens  : {usage.prompt_token_count:,}")
        print(f"  Output tokens : {usage.candidates_token_count:,}")
        print(f"  Total tokens  : {usage.total_token_count:,}")

    except Exception as e:
        print(f"Error processing {cv_name}: {e}")

print("\nDONE!")

### **OpenAI**



In [ ]:
for cv_name, pages in cv_groups.items():
    try:
        print(f"\n→ Processing CV: {cv_name} ({len(pages)} pages)")

        image_inputs = []
        for img_path in pages:
            base64_data_url = image_to_data_url(img_path)
            image_inputs.append({
                "type": "input_image",
                "image_url": base64_data_url
            })

        max_tokens = min(12000, 4000 + 2000 * len(pages))

        response_openai = client_openai.responses.parse(
            model="gpt-4.1-mini",
            input=[
                {
                    "role": "user",
                    "content": [
                        {"type": "input_text", "text": prompt_context_between_pages},
                        *image_inputs
                    ]
                }
            ],
            max_output_tokens=max_tokens,
            text_format=ResumeData
        )
        try:
            print("\n--- TOKEN USAGE ---")
            print(response_openai.usage)
        except:
            print("No usage info available")

        resume_data = response_openai.output_parsed

        output_path = OUTPUT_DIR_OPENAI / f"{cv_name}.json"
        with open(output_path, "w", encoding="utf-8") as f:
            json.dump(resume_data.model_dump(), f, indent=2, ensure_ascii=False)
        print(f"Saved {output_path.name}")
    except Exception as e:
        print(f"✗ Error processing {cv_name}: {e}")

print("\nDONE!")


### **LiquidAI**